In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3



In [2]:
# lists 
titles= []
prices= []
ratings=[]
instock= []
URLs= []

In [3]:
base_url = "https://books.toscrape.com/"

for page in range(1,6):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    src = response.content
    soup =BeautifulSoup(src , "html.parser")
    book_titles = soup.find_all("h3")
    book_prices = soup.find_all("p" , {"class":"price_color"})
    book_ratings = soup.find_all("p" , {"class":"star-rating"})
    instock_availability = soup.find_all("p" , {"class":"instock availability"})
    book_urls = soup.find_all("div" , {"class":"image_container"})
    relative_url = book_urls[0].a.get("href")
    
    for item in range(len(book_titles)):
        title = book_titles[item].a.get("title")
        titles.append(title)
        price = book_prices[item].text
        prices.append(price)
        rating= book_ratings[item].get("class")[1]
        ratings.append(rating)
        stock= instock_availability[item].text.strip()
        instock.append(stock)
        url=base_url+ book_urls[item].a.get("href")
        URLs.append(url)




In [4]:

print (len(titles))
print (len(prices))
print (len(ratings))
print (len(instock))
print (len(URLs))

100
100
100
100
100


In [5]:
data = {"title"  :titles,
        "price"  :prices,
        "rating" :ratings,
        "instock":instock,
        "URL"    : URLs
        }

In [6]:
df =pd.DataFrame(data)
df.head(10)

,title,price,rating,instock,URL
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/a-light-in-the-atti...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/tipping-the-velvet_...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/soumission_998/inde...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/sharp-objects_997/i...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/sapiens-a-brief-his...
5,The Requiem Red,£22.65,One,In stock,https://books.toscrape.com/the-requiem-red_995...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four,In stock,https://books.toscrape.com/the-dirty-little-se...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three,In stock,https://books.toscrape.com/the-coming-woman-a-...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four,In stock,https://books.toscrape.com/the-boys-in-the-boa...
9,The Black Maria,£52.15,One,In stock,https://books.toscrape.com/the-black-maria_991...


In [7]:
df["price"] = df["price"].str.replace("£", "").astype(float)

In [8]:
df["price"].dtype

dtype('float64')

In [9]:
df["instock"].value_counts()

instock
In stock    100
Name: count, dtype: int64

In [12]:
df["instock"] = df["instock"].map({"In stock"     : True,
                                    "Out of stock": False})

In [14]:
df["rating"].unique()

array(['Three', 'One', 'Four', 'Five', 'Two'], dtype=object)

In [16]:
df["rating"] = df["rating"].map({"Three":  3,
                    "One" : 1,
                    "Four": 4,
                    "Five": 5,
                    "Two" : 2
                    })

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   title    100 non-null    object 
 1   price    100 non-null    float64
 2   rating   100 non-null    int64  
 3   instock  100 non-null    bool   
 4   URL      100 non-null    object 
dtypes: bool(1), float64(1), int64(1), object(2)
memory usage: 3.4+ KB


In [18]:
df.to_csv("books.csv", index=False)

In [19]:
import sqlite3

connection = sqlite3.connect("books.db")

In [20]:
df.to_sql("books", connection, if_exists="replace", index=False)

100

In [21]:
pd.read_sql("SELECT * FROM books LIMIT 5", connection)

,title,price,rating,instock,URL
0,A Light in the Attic,51.77,3,1,https://books.toscrape.com/a-light-in-the-atti...
1,Tipping the Velvet,53.74,1,1,https://books.toscrape.com/tipping-the-velvet_...
2,Soumission,50.10,1,1,https://books.toscrape.com/soumission_998/inde...
3,Sharp Objects,47.82,4,1,https://books.toscrape.com/sharp-objects_997/i...
4,Sapiens: A Brief History of Humankind,54.23,5,1,https://books.toscrape.com/sapiens-a-brief-his...


In [22]:
query = """
SELECT rating, AVG(price) AS average_price
FROM books
GROUP BY rating;
"""

result = pd.read_sql(query, connection)
result

,rating,average_price
0,1,35.519545
1,2,35.909474
2,3,36.839091
3,4,33.981667
4,5,30.012105


In [23]:
query = """
SELECT *
FROM books
WHERE rating IN (4, 5)
ORDER BY price DESC
LIMIT 5;
"""

result = pd.read_sql(query, connection)
result

,title,price,rating,instock,URL
0,The Death of Humanity: and the Case for Life,58.11,4,1,https://books.toscrape.com/the-death-of-humani...
1,The Past Never Ends,56.50,4,1,https://books.toscrape.com/the-past-never-ends...
2,Sapiens: A Brief History of Humankind,54.23,5,1,https://books.toscrape.com/sapiens-a-brief-his...
3,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1,https://books.toscrape.com/scott-pilgrims-prec...
4,Behind Closed Doors,52.22,4,1,https://books.toscrape.com/behind-closed-doors...


In [25]:
query = """
SELECT rating, COUNT(*) AS out_of_stock
FROM books
WHERE instock = 0
GROUP BY rating;
"""

result = pd.read_sql(query, connection)
result

,rating,out_of_stock


In [26]:
connection.close()